# MNIST CNN Training 


## 1. Model Definition :computer:

In [1]:
import torch
from torch import nn

class MnistModel1(nn.Module):
    """CNN model with two convolutional/max-pooling blocks and a classification head."""
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
        super().__init__()
        
        # The MNIST images are 28x28. The size progression is:
        # 28x28 -> Conv2d(k=3, p=1) -> 28x28 -> MaxPool2d(k=2, s=2) -> 14x14
        self.CNN_block1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        
        # 14x14 -> Conv2d(k=3, p=1) -> 14x14 -> MaxPool2d(k=2, s=2) -> 7x7
        self.CNN_block2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # Flattening and Linear layer for classification
        # Input size: hidden_units * 7 * 7
        self.Classification = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units * 7 * 7, out_features=output_shape),
        )

    def forward(self, x: torch.Tensor):
        x = self.CNN_block1(x)
        x = self.CNN_block2(x)
        x = self.Classification(x)
        return x

d:\Projects\CNN_Model\.venv\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


## 2. Helper Functions :wrench:

In [2]:
import torch

def print_train_time(start: float, end: float, device=None):
    """Prints difference between start and end time."""
    total_time = end - start
    print(f"\nTrain time on {device}: {total_time:.3f} seconds")
    return total_time

def accuracy_fn(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    """Calculates accuracy between truth labels and predicted labels."""
    # Assumes y_pred is the argmax of logits or already the predicted class index
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = (correct / len(y_pred)) * 100
    return acc

## 3. Training Function :muscle:

In [ ]:
import torch
from torch import nn
from tqdm.auto import tqdm
from timeit import default_timer as timer
import matplotlib.pyplot as plt

train_losses = []
test_losses = []
test_accuracies = []

def train_and_save(model_0: nn.Module, train_dataloader: torch.utils.data.DataLoader, test_dataloader: torch.utils.data.DataLoader, epochs: int, loss_fn: nn.Module, optimizer: torch.optim.Optimizer, device="cuda") -> nn.Module:
    """Trains and tests a PyTorch model and saves the best one."""
    model_0.to(device)
    
    start_time = timer()

    for epoch in tqdm(range(epochs)):
        print(f"Epoch: {epoch}\n-------")
        train_loss = 0
        model_0.train()

        for batch, (X, y) in enumerate(train_dataloader):
            X, y = X.to(device), y.to(device)
            
            # 1. Forward pass
            y_pred = model_0(X)
            
            # 2. Calculate loss
            loss = loss_fn(y_pred, y)
            train_loss += loss.item()
            
            # 3. Optimizer zero grad
            optimizer.zero_grad()
            
            # 4. Loss backward
            loss.backward()
            
            # 5. Optimizer step
            optimizer.step()

            if batch % 400 == 0:
                print(f"Looked at {batch * len(X)}/{len(train_dataloader.dataset)} samples")

        # Average train loss per epoch
        train_loss /= len(train_dataloader)
        train_losses.append(train_loss)

        ### Testing
        test_loss, test_acc = 0, 0
        model_0.eval()
        with torch.inference_mode():
            for X, y in test_dataloader:
                X, y = X.to(device), y.to(device)
                
                # 1. Forward pass
                test_pred_logits = model_0(X)
                
                # 2. Calculate loss and accuracy
                test_loss += loss_fn(test_pred_logits, y).item()
                test_acc += accuracy_fn(y_true=y, y_pred=test_pred_logits.argmax(dim=1))

            # Average test loss and accuracy per epoch
            test_loss /= len(test_dataloader)
            test_acc /= len(test_dataloader)

        test_losses.append(test_loss)
        test_accuracies.append(test_acc)

        print(f"\nTrain loss: {train_loss:.5f} | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%\n")
        
        # Save the model if it achieves high accuracy (98.5%)
        if test_acc >= 98.50:
            torch.save(model_0.state_dict(), "mnist_model.pth")
            print(f"Model saved at epoch {epoch} with accuracy {test_acc:.2f}%")
    
    end_time = timer()
    print_train_time(start_time, end_time, device=device)
    
    return model_0

## 4. Data Loading and Model Training Setup :page_with_curl:

In [ ]:
import torch
from torch import nn
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Hyperparameters
epochs = 10 # Set to 50 for full training, starting with 10 for a quicker run
BATCH_SIZE = 32
learning_rate = 0.01
hidden_units = 20

# Set random seed for reproducibility
torch.manual_seed(42)

# Loss function
loss_fn = nn.CrossEntropyLoss()

# Load MNIST dataset
train_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(), 
    target_transform=None
)
class_names = train_data.classes

test_data = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

# Create data loaders
train_dataloader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

test_dataloader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print(f"Number of training samples: {len(train_data)}")
print(f"Number of test samples: {len(test_data)}")
print(f"Classes: {class_names}")

## 5. Initialize Model and Optimizer :sparkles:

In [ ]:
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Initialize model
model1 = MnistModel1(
    input_shape=1, # 1 color channel (grayscale)
    hidden_units=hidden_units,
    output_shape=len(class_names) # 10 output classes (digits 0-9)
).to(device)

# Initialize optimizer
optimizer = torch.optim.SGD(params=model1.parameters(), lr=learning_rate)

print(f"\nModel architecture:")
print(model1)

## 6. Train the Model :chart_with_upwards_trend:

In [ ]:
# Train the model
NewModel = train_and_save(
    model1,
    train_dataloader,
    test_dataloader,
    epochs,
    loss_fn,
    optimizer,
    device=device
)

## 7. Visualize Training Results :bar_chart:

In [ ]:
# Plot training and test loss
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Test Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(test_accuracies, label='Test Accuracy', color='green')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Test Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"\nFinal Test Accuracy: {test_accuracies[-1]:.2f}%")

## 8. Test Predictions on Sample Images :mag:

In [ ]:
# Make predictions on a few test samples
NewModel.eval()
with torch.inference_mode():
    # Get a batch of test images
    test_images, test_labels = next(iter(test_dataloader))
    test_images, test_labels = test_images.to(device), test_labels.to(device)
    
    # Make predictions
    test_preds = NewModel(test_images)
    pred_labels = test_preds.argmax(dim=1)

# Visualize predictions
plt.figure(figsize=(12, 8))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(test_images[i].cpu().squeeze(), cmap='gray')
    # Check if prediction is correct
    is_correct = pred_labels[i] == test_labels[i]
    color = "green" if is_correct else "red"
    plt.title(f"Pred: {pred_labels[i].cpu()} | True: {test_labels[i].cpu()}", color=color)
    plt.axis('off')
plt.tight_layout()
plt.show()